# EarthForge — Getting Started

This notebook demonstrates the core EarthForge workflow:
1. Format detection — inspect any geospatial file without knowing its format
2. Raster info — read COG metadata via range requests
3. Vector info — inspect GeoParquet schema and bounds
4. STAC search — discover imagery from public catalogs

No data is downloaded until you explicitly call `stac fetch` or `raster convert`.

In [ ]:
# Install if needed
# !pip install earthforge[all]

In [ ]:
import sys
sys.path.insert(0, '../../packages/core/src')
sys.path.insert(0, '../../packages/raster/src')
sys.path.insert(0, '../../packages/vector/src')
sys.path.insert(0, '../../packages/stac/src')

from earthforge.core.config import EarthForgeProfile

# Default profile — Earth Search STAC API
profile = EarthForgeProfile(
    name='default',
    stac_api='https://earth-search.aws.element84.com/v1',
    storage_backend='local',
)
print('EarthForge ready')

## 1. Format Detection

`detect()` uses a three-stage chain: magic bytes → file extension → content inspection.
It reads only the first 512 bytes of a local file or remote URL.

In [ ]:
from earthforge.core.formats import detect

# Test with a public Copernicus DEM COG
cop_dem_url = (
    'https://copernicus-dem-30m.s3.amazonaws.com/'
    'Copernicus_DSM_COG_10_N37_00_W085_00_DEM/'
    'Copernicus_DSM_COG_10_N37_00_W085_00_DEM.tif'
)

fmt = await detect(cop_dem_url)
print(f'Detected format: {fmt}')

## 2. Raster Info

Read COG metadata — dimensions, CRS, band count, overviews — without downloading the file.

In [ ]:
from earthforge.raster.info import inspect_raster

info = await inspect_raster(cop_dem_url)

print(f'Dimensions: {info.width} x {info.height}')
print(f'CRS:        {info.crs}')
print(f'Bands:      {info.band_count}')
print(f'Tiled:      {info.is_tiled} ({info.tile_width}x{info.tile_height})')
print(f'Overviews:  {info.overview_count} ({info.overview_levels})')
print(f'Compress:   {info.compression}')
print(f'Bounds:     {info.bounds}')

In [ ]:
from earthforge.raster.validate import validate_cog

val = await validate_cog(cop_dem_url)
print(f'Valid COG: {val.is_valid}')
for check in val.checks:
    status = 'PASS' if check.passed else 'FAIL'
    print(f'  [{status}] {check.name}: {check.message}')

## 3. STAC Search

Search Element84 Earth Search for Sentinel-2 imagery over Kentucky.

In [ ]:
from earthforge.stac.search import search_catalog

results = await search_catalog(
    profile,
    collections=['sentinel-2-l2a'],
    bbox=[-85.0, 37.0, -84.0, 38.0],
    datetime_range='2025-06-01/2025-09-30',
    max_items=5,
)

print(f'Found {len(results.items)} items')
for item in results.items:
    print(f'  {item.id} | {item.datetime} | {item.asset_count} assets')

In [ ]:
# Inspect the first item's assets
if results.items:
    item = results.items[0]
    print(f'Item: {item.id}')
    print(f'Bbox: {item.bbox}')
    print()
    for asset in item.assets[:8]:
        print(f'  {asset.key:<10} {asset.media_type or ""}')

## 4. Output as JSON

All EarthForge results are Pydantic models — serialize to JSON for downstream tools.

In [ ]:
import json

# Serialize the raster info result
info_json = json.loads(info.model_dump_json())
print(json.dumps(info_json, indent=2))